# AXI test 22: the long session on Gemini 3.8 Flash, counted the way a company that runs the model counts
**What this is.** One hosted model, **Gemini 3.8 Flash**, given heavy turns (every request comes with a block of operations-log noise, about 40 lines) over a long job, in two ways: **alone** (the whole chat kept and sent again at every turn, until the chat nears the model's 1,000,000-token window) and **with AXI** (the exact state held by the framework and shown at every step; nothing kept). Measured for both: **drift and hallucination** (does a reply give the exact next state; can it be applied at all), **speed** (steps and exact steps per hour of calls, by context size), **latency** (seconds per call and seconds to the first token, by context size), **tokens** (prompt tokens, how many of them the service served from its cache, reply and thinking tokens), and the **energy stand-in** (computed tokens and seconds: the API gives no joules for Google's hardware, and the page says so). A third part, the **depth probe**, gives the model a PERFECT history at several depths, to tell "the chat is long" from "its own earlier mistakes are in the chat".

The predictions and the reading rules were written before any call in `docs/GEMINI_LONG_TEST.md` (hash-locked).

**This test costs real money, and this notebook never turns billing on.** You do that in Google AI Studio. Then: (1) read the price of Gemini 3.8 Flash on the AI Studio price page (dollars per million input tokens, per million output tokens, and per million *cached* input tokens if the page lists one) and type them in Cell 5; (2) run the pilot in Cell 6, which sends only 20 real turns of each arm and prints what the whole test would cost **at your prices**; (3) only then type a budget in dollars (`BUDGET_USD`) and run the rest. The notebook adds up what the saved answers cost and **refuses the next call once the total reaches the budget**; a second guard stops each run at a cap on tokens sent. Everything finished is saved and the same cell carries on.

Run each cell **one at a time, top to bottom**. **Keep this tab open** while a run is going. If it stops for any reason, open the notebook again, run Cells 2 to 5, then run the cell you were on again.

### Cell 1: put your key into Colab Secrets (once; no code to run)
Colab left bar -> key icon (*Secrets*) -> **Add new secret**. Name: `GOOGLE_API_KEY` (exactly). Value: your key from `aistudio.google.com` -> *API Keys*. Switch **Notebook access** on. Never paste the key into a cell or into the chat.

### Cell 2: get the code from GitHub (finish the update 73 commands in Codespaces and `git push` first)

In [ ]:
!rm -rf /content/axi && git clone -q --depth 1 https://github.com/gaurshalendra2412/axi-runtime /content/axi
import os
assert os.path.exists("/content/axi/python/axi/experiments/gemini_long.py"), "the code is not on GitHub yet: finish the update 73 commands (git push) first"
print("code ok")

### Cell 3: install the Gemini library (about 20 seconds)

In [ ]:
!pip -q install -U google-genai

### Cell 4: connect Google Drive (a pop-up asks permission: Connect / Allow). Your progress and every reply are kept there

In [ ]:
import sys
sys.path.insert(0, "/content/axi/python")
from colab.colab_persist import open_store
store = open_store("/content/out", "axi_gemini_long")

### Cell 5: settings. Fill the lines at the top. PAID is True only after you turned billing on. The budget stays empty until you have read the pilot

In [ ]:
import json, time
from collections import deque
from google.colab import userdata, files
from axi.experiments import gemini_long as G
from axi.experiments.api_llm import QuotaStop, ServiceBusy, BudgetStop

PAID = None          # True ONLY after you turned billing on for this key in Google AI Studio. False = a free key (about 20 calls a day: not enough for the pilot)
PRICE_IN = None      # dollars per 1,000,000 INPUT tokens of Gemini 3.8 Flash, from the AI Studio price page
PRICE_OUT = None     # dollars per 1,000,000 OUTPUT tokens (thinking tokens are billed as output), same page
PRICE_CACHED = None  # dollars per 1,000,000 CACHED input tokens, if the page lists one; leave None if it does not
BUDGET_USD = None    # the most you allow this test to cost. Leave None until you have read the pilot in Cell 6. The notebook stops by itself at this amount.
THINKING = None      # None = the model's own default

assert PAID in (True, False), "Set PAID: True if you turned billing on, False for a free key."
try: KEY = userdata.get("GOOGLE_API_KEY")
except Exception: raise SystemExit("No key found. Do Cell 1: a Secret named GOOGLE_API_KEY with Notebook access on.")
if PAID: assert PRICE_IN and PRICE_OUT, "Type PRICE_IN and PRICE_OUT (dollars per million tokens) from the AI Studio price page. The notebook has no built-in price on purpose."
TAG = str(THINKING or "default"); CACHE = "axi_gemini_long_cache_%s.jsonl" % TAG
print("reply file:", store.restore(CACHE))
guard = dict(budget_usd=BUDGET_USD, price_in=PRICE_IN, price_out=PRICE_OUT, price_cached=PRICE_CACHED) if (PAID and BUDGET_USD) else dict(price_in=PRICE_IN, price_out=PRICE_OUT, price_cached=PRICE_CACHED)
llm, MODEL = G.build_chat_llm(KEY, store.path(CACHE), daily_limit=None if PAID else 20, min_interval=0.3 if PAID else 4.5, on_write=lambda: store.persist(CACHE), thinking=THINKING, **guard)
META = dict(model=MODEL, thinking=THINKING)
sp = llm.spent_usd()
print("paid:", PAID, "| model:", MODEL, "| thinking:", THINKING or "model default", "| budget:", ("$%.2f" % BUDGET_USD) if BUDGET_USD else "NOT SET (pilot only)", "| spent so far with these settings:", ("$%.2f" % sp) if sp is not None else "-")
RESET = "The free quota resets at midnight Pacific time. Then open this notebook again, run Cells 2 to 5, and run this cell again."
BUSY = "This is Google's server being busy. Wait at least 30 minutes, then run THIS SAME CELL once. Do not run it again and again."
def go(fn):
    try: return fn()
    except QuotaStop as e:
        print("\nSTOPPED FOR NOW: " + str(e))
        if isinstance(e, BudgetStop): pass
        elif isinstance(e, ServiceBusy): print(BUSY)
        else: print(RESET)
        return None
def need_budget(): assert PAID and BUDGET_USD, "Read the pilot (Cell 6), then type BUDGET_USD in Cell 5, run Cell 5 again, and run this cell again."
def save(res, name):
    p = store.path(name + ".json"); json.dump(res, open(p, "w")); store.persist(name + ".json"); return p
def persisting(name, inner):
    print("progress file:", store.restore(name + ".jsonl"))
    def cb(*a):
        inner(*a)
        if a[0] % 25 == 0 or a[0] == a[1]: store.persist(name + ".jsonl")
    return cb
def prog_k(t, n, r):
    if t % 10 == 0 or t == n:
        sp = llm.spent_usd(); print(f"turn {t}/{n} | prompt {r['prompt_tokens']:,} (cached {r['cache_tokens']:,}) | {r['seconds']:.1f} s, first token {r['first_token'] if r['first_token'] is None else round(r['first_token'], 2)} s | exact {r['reply_exact']} | spent {('$%.2f' % sp) if sp is not None else '-'}")
def prog_a(t, n, r):
    if t % 100 == 0 or t == n:
        sp = llm.spent_usd(); print(f"step {t}/{n} | prompt {r['prompt_tokens']:,} | {r['seconds']:.1f} s | held exact {r['exact']} | spent {('$%.2f' % sp) if sp is not None else '-'}")

### Cell 6: the PILOT. 20 real turns of each arm (a few cents). The program answers the first 200 steps of the AXI arm itself so that the graph has its full size. It prints the real size of one heavy turn, the speed and latency, and the projected cost of the whole test at your prices

In [ ]:
assert PAID, "The pilot needs a paid key (about 40 calls)."
pk = go(lambda: G.run_kept(llm, G.SEED_PILOT, 20, store.path("gl_pilot_k.jsonl"), meta=META))
pa = go(lambda: G.run_axi(llm, G.SEED_PILOT, 220, store.path("gl_pilot_a.jsonl"), meta=META, scripted=200))
if pk and pa:
    store.persist("gl_pilot_k.jsonl"); store.persist("gl_pilot_a.jsonl")
    a_live = pa["rows"][200:]
    G.print_stats("pilot K (kept chat)", pk["rows"]); G.print_stats("pilot A (with AXI)", a_live)
    rep = G.pilot_report(pk["rows"], a_live, PRICE_IN, PRICE_OUT, PRICE_CACHED)
    print()
    for k, v in rep.items(): print(f"  {k}: {v if not isinstance(v, float) else round(v, 3)}")
    print("\nRead these before the budget: k_growth_per_turn is the real size of one heavy turn; k_turns_that_fit is how many turns fit under 950,000 tokens; cost_k is arm K with the cache share measured now (cost_k_nocache: with no cache discount); cost_a is arm A (1,500 steps).")
    print("Then type BUDGET_USD in Cell 5 (a little above cost_a + cost_k + the depth probe, or lower: the run stops by itself and what it finished stays valid).")

### Cell 7: ARM A, with AXI. 1,500 steps, the exact state shown at every step with the same heavy turns. A line every 100 steps

In [ ]:
need_budget()
nameA = "gl_A_seed%d_%s" % (G.SEED, TAG)
resA = go(lambda: G.run_axi(llm, G.SEED, G.A_STEPS, store.path(nameA + ".jsonl"), meta=META, max_tokens=G.A_TOKEN_CAP, progress=persisting(nameA, prog_a)))
if resA:
    G.print_stats("ARM A (with AXI)", resA["rows"]); G.print_bands(resA["rows"]); store.persist(nameA + ".jsonl")
    print("spent so far on this model: $%.2f" % llm.spent_usd()); files.download(save(resA, nameA))

### Cell 8: ARM K, the kept chat, alone. Up to 700 turns, it stops when the next prompt would pass 950,000 tokens. A line every 10 turns. This is the long and costly cell; if it stops (budget, busy server, tab closed), run Cells 2 to 5 and this cell again

In [ ]:
need_budget()
nameK = "gl_K_seed%d_%s" % (G.SEED, TAG)
resK = go(lambda: G.run_kept(llm, G.SEED, G.K_MAX_STEPS, store.path(nameK + ".jsonl"), meta=META, max_tokens=G.K_TOKEN_CAP, progress=persisting(nameK, prog_k)))
if resK:
    print("stopped because:", resK["stopped"] or "the 700 turns were done")
    G.print_stats("ARM K (kept chat)", resK["rows"]); G.print_bands(resK["rows"]); store.persist(nameK + ".jsonl")
    print("spent so far on this model: $%.2f" % llm.spent_usd()); files.download(save(resK, nameK))

### Cell 9: the DEPTH PROBE. A perfect history (the ideal replies) at depths 50, 100, 200, 300 and 350, then 10 real turns at each. Cheaper than arm K; it separates the length of the chat from the chat's own mistakes

In [ ]:
need_budget()
nameD = "gl_D_seed%d_%s" % (G.SEED, TAG)
resD = go(lambda: G.run_depth(llm, G.SEED, G.D_STARTS, G.D_ASKS, store.path(nameD + ".jsonl"), meta=META, max_tokens=G.D_TOKEN_CAP, progress=lambda s, t, r: print(f"depth {s}, turn {t}: prompt {r['prompt_tokens']:,} | exact {r['reply_exact']} | {r['seconds']:.1f} s") if t % 5 == 0 else None))
if resD:
    store.persist(nameD + ".jsonl")
    for s, v in resD["summary"].items(): print(f"start {s}: prompt about {v['mean_prompt']:,.0f} | exact {100*v['reply_exact']:.0f}% | applicable {100*v['reply_ok']:.0f}% | {v['seconds_per_step']:.1f} s | first token p50 {v['first_p50']}")
    print("spent so far on this model: $%.2f" % llm.spent_usd()); files.download(save(resD, nameD))

### Cell 10: the READING. Replays every saved reply (a check that nothing was changed), prints the speed and latency tables and the 20 lines with yes / NO, and saves the report. Send me the output and the downloaded files

In [ ]:
ok = {}
try: G.replay(resK["rows"], G.SEED, "K"); ok["K"] = True
except Exception as e: print("replay K:", e); ok["K"] = False
try: G.replay(resA["rows"], G.SEED, "A"); ok["A"] = True
except Exception as e: print("replay A:", e); ok["A"] = False
if resD: G.replay_depth(resD["rows"], G.SEED)
checks = G.verdict(resK, resA, resD, ok)
print("ARM K, by the size of the prompt:"); G.print_bands(resK["rows"])
print("\nARM A, by tenths of the job:")
for t in G.tenths(resA["rows"]): print(f"  steps {t['first']}-{t['last']}: held exact {100*t['held_exact']:.1f}% | {t['seconds_per_step']:.2f} s | first token p50 {t['first_p50']} | prompt {t['prompt_per_step']:,.0f}")
G.print_verdict(checks)
print("usage:", llm.usage(), "| spent: $%.2f" % llm.spent_usd())
files.download(save(dict(K=resK, A=resA, D=resD, checks=checks, replays=ok), "gemini_long_all_%s" % TAG))

### Done
Files downloaded: the arm A, arm K, depth and `gemini_long_all_*.json` files and the `.jsonl` reply file. Upload `gemini_long_all_default.json` to the chat. Nothing here says anything about any company's own service: it measures this model on this job.